# UbiComb plant: residual hybrid benchmark on Kaggle

This notebook trains the Short-Range Expert and Long-Context Expert on the published UbiComb plant training set, fits the residual fusion from protein-grouped out-of-fold predictions, refits both experts, and compares with the authors' dataset-specific checkpoint on matched published test sites.

Select **GPU T4 x2** and enable **Internet** in Kaggle settings. Add a Kaggle secret named `GITHUB_TOKEN` with read access to the private repository; `HF_TOKEN` is optional. Training uses five outer folds and 15 epochs per candidate without early stopping. Separate fold workers use the two T4s concurrently; the final refit uses one T4. The released dataset files come from the Git submodule, and the public checkpoint is downloaded and checksum-verified below.

Enable Kaggle notebook **Persistence → Files only** before training. Intermediate files are written to `/kaggle/working`; for a different account or notebook, attach a saved output dataset and set `RESUME_RUN_DIR` in the paths cell. Kaggle persistence is best effort, so save a version or export the run directory at milestones.

## Repository and credentials

In [1]:
from pathlib import Path
import os, subprocess, sys

from kaggle_secrets import UserSecretsClient

def secret(name):
    try:
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

GITHUB_TOKEN = secret('GITHUB_TOKEN')
HF_TOKEN = secret('HF_TOKEN')
if not GITHUB_TOKEN:
    raise RuntimeError('Add the read-only GITHUB_TOKEN secret to this Kaggle notebook.')

REPO_ROOT = Path('/kaggle/working/Ubiquitination-Sites-Prediction')
REPO_REF = None  # Set to the printed commit SHA when restoring a prior run.
git_env = os.environ.copy()
git_env['GIT_TERMINAL_PROMPT'] = '0'
git_env['GITHUB_TOKEN'] = GITHUB_TOKEN
askpass = Path('/kaggle/working/ubipred_git_askpass.sh')
askpass.write_text('#!/bin/sh\ncase "$1" in *Username*) echo x-access-token;; *) echo "$GITHUB_TOKEN";; esac\n')
askpass.chmod(0o700)
try:
    if not (REPO_ROOT / '.git').exists():
        subprocess.run(['git', 'clone', '--recurse-submodules',
                        'https://github.com/kaleabnega/Ubiquitination-Sites-Prediction.git',
                        str(REPO_ROOT)], check=True, env=git_env)
    elif REPO_REF is None:
        subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True, env=git_env)
    if REPO_REF is not None:
        subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', '--detach', REPO_REF],
                       check=True, env=git_env)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'submodule', 'update', '--init', '--recursive'],
                   check=True, env=git_env)
finally:
    askpass.unlink(missing_ok=True)
os.chdir(REPO_ROOT)
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HF_HOME'] = '/kaggle/working/ubipred_huggingface_cache'
Path(os.environ['HF_HOME']).mkdir(parents=True, exist_ok=True)
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)

Cloning into '/kaggle/working/Ubiquitination-Sites-Prediction'...
Submodule 'replication/MMUbiPred' (https://github.com/PakhrinLab/MMUbiPred.git) registered for path 'replication/MMUbiPred'
Cloning into '/kaggle/working/Ubiquitination-Sites-Prediction/replication/MMUbiPred'...


Submodule path 'replication/MMUbiPred': checked out 'fb45202d0a91c58e74d73a9f8e4ad43dff4e48e5'
2e8dca39ee5589ba2f2e51160fe7be7070243d7a


CompletedProcess(args=['git', 'rev-parse', 'HEAD'], returncode=0)

## Dependencies and accelerator check

In [2]:
%pip install -q -r experiment/requirements-colab.txt h5py gdown
%pip uninstall -y torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 92.3 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
Note: you may need to restart the kernel to use updated packages.


In [3]:
import json, torch

if not torch.cuda.is_available() or torch.cuda.device_count() != 2:
    raise RuntimeError('Select Kaggle GPU T4 x2 before training.')
print('Visible GPUs:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

def run_live(command, env=None):
    command = [str(part) for part in command]
    print('$', ' '.join(command), flush=True)
    process = subprocess.Popen(command, env=env, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()
    if code:
        raise subprocess.CalledProcessError(code, command)

Visible GPUs: ['Tesla T4', 'Tesla T4']


## Paths, source preflight and released checkpoint

The released source files are already in the Git submodule. The checkpoint is taken from an attached Kaggle dataset if present, or downloaded from the authors' public Drive folder. Its SHA-256 is checked before use. The independent test records are not loaded during this audit.

In [4]:
import hashlib, shutil

CONFIG = Path('experiment/configs/benchmarks/ubicomb_residual_v1.json')
BENCHMARK = 'ubicomb'
RUN_DIR = Path('/kaggle/working/mmubipred_benchmark_runs') / BENCHMARK / 'run'
SOURCE_DIR = Path('/kaggle/working/mmubipred_benchmark_sources') / BENCHMARK
MODEL_FILE = SOURCE_DIR / 'UBICOMB_DeepUBI_AAindex_One_Hot_ProtT5_drop_out_Changed3257.h5'
MODEL_DRIVE_ID = '1jGURx76ll40d0TebNO9CQOcG9SF8V0xT'
MODEL_SHA256 = 'a1057c0e13d6dbd1348f83b26ed0fa347498dec914b7c671f274713fdc6cdefd'

# To restore a previously saved Kaggle output, attach it as Input and set this path
# to the directory ending in /ubicomb/run. Leave None for a new or persisted session.
RESUME_RUN_DIR = None
if RESUME_RUN_DIR is not None:
    previous = Path(RESUME_RUN_DIR)
    if not (previous / 'protocol.json').is_file():
        raise FileNotFoundError(f'Prior run protocol missing: {previous}')
    if RUN_DIR.exists() and any(RUN_DIR.iterdir()):
        raise RuntimeError(f'Current run directory is not empty: {RUN_DIR}')
    shutil.copytree(previous, RUN_DIR, dirs_exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)
SOURCE_DIR.mkdir(parents=True, exist_ok=True)

def sha256(path):
    value = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            value.update(block)
    return value.hexdigest()

if not MODEL_FILE.is_file():
    attached = list(Path('/kaggle/input').rglob(MODEL_FILE.name))
    if len(attached) > 1:
        raise RuntimeError(f'Multiple attached copies of the baseline: {attached}')
    if attached:
        shutil.copy2(attached[0], MODEL_FILE)
    else:
        import gdown
        gdown.download(id=MODEL_DRIVE_ID, output=str(MODEL_FILE), quiet=False)
if not MODEL_FILE.is_file() or sha256(MODEL_FILE) != MODEL_SHA256:
    raise RuntimeError(f'Baseline checkpoint missing or checksum mismatch: {MODEL_FILE}')
print('Verified checkpoint:', MODEL_FILE)
print('Run directory:', RUN_DIR)

def stage_command(name, *extra):
    return [sys.executable, '-u', 'experiment/scripts/run_dataset_benchmark.py', name,
            '--config', CONFIG, '--run-dir', RUN_DIR, '--source-dir', SOURCE_DIR,
            '--model-file', MODEL_FILE, *extra]

def stage(name, *extra):
    run_live(stage_command(name, *extra))

stage('audit')
audit = json.loads((RUN_DIR / 'source_audit.json').read_text())
display(audit)
assert audit['source_preflight_passed'], audit['blockers']

Downloading...
From: https://drive.google.com/uc?id=1jGURx76ll40d0TebNO9CQOcG9SF8V0xT
To: /kaggle/working/mmubipred_benchmark_sources/ubicomb/UBICOMB_DeepUBI_AAindex_One_Hot_ProtT5_drop_out_Changed3257.h5
100%|██████████| 36.6M/36.6M [00:00<00:00, 51.6MB/s]


Verified checkpoint: /kaggle/working/mmubipred_benchmark_sources/ubicomb/UBICOMB_DeepUBI_AAindex_One_Hot_ProtT5_drop_out_Changed3257.h5
Run directory: /kaggle/working/mmubipred_benchmark_runs/ubicomb/run
$ /usr/bin/python3 -u experiment/scripts/run_dataset_benchmark.py audit --config experiment/configs/benchmarks/ubicomb_residual_v1.json --run-dir /kaggle/working/mmubipred_benchmark_runs/ubicomb/run --source-dir /kaggle/working/mmubipred_benchmark_sources/ubicomb --model-file /kaggle/working/mmubipred_benchmark_sources/ubicomb/UBICOMB_DeepUBI_AAindex_One_Hot_ProtT5_drop_out_Changed3257.h5
{
  "benchmark": "ubicomb",
  "training_started": false,
  "test_predictions_generated": false,
  "files": {
    "train": [
      {
        "filename": "UbiComb_training_data.txt",
        "available": true,
        "sha256": "516206065e66dc7a9fd94e51d5657011924576ff75327197b48804364a1fc278"
      }
    ],
    "test": [
      {
        "filename": "Positive_UbiCom_testing.fasta",
        "available": 

{'baseline': {'available': True,
  'dataset_assignment_evidence': 'authors upstream README/notebook filename mapping',
  'expected_filename': 'UBICOMB_DeepUBI_AAindex_One_Hot_ProtT5_drop_out_Changed3257.h5',
  'input_shapes': [[None, 49, 31], [None, 49, 21], [None, 49]],
  'sha256': 'a1057c0e13d6dbd1348f83b26ed0fa347498dec914b7c671f274713fdc6cdefd'},
 'benchmark': 'ubicomb',
 'blockers': [],
 'files': {'test': [{'available': True,
    'filename': 'Positive_UbiCom_testing.fasta',
    'sha256': '7489012cf71b7173bc6e26264aed6cce531101dd494c95e4acbdd208a38719af'},
   {'available': True,
    'filename': 'Negative_UbiCom_testing.fasta',
    'sha256': '8d7dc133fcf70e70c87c845977270088b2ed99ec72e4c0093ad56f4718f7fd0a'}],
  'train': [{'available': True,
    'filename': 'UbiComb_training_data.txt',
    'sha256': '516206065e66dc7a9fd94e51d5657011924576ff75327197b48804364a1fc278'}]},
 'source_preflight_passed': True,
 'source_url': 'https://drive.google.com/drive/folders/1Bcx55mlv4FKqK2XVTww8XBaNq

## Prepare training context

The source adapter validates the published training windows against UniProt protein sequences and requires at least 95% usable context. It does not access the published test split.

In [5]:
run_live([sys.executable, '-m', 'unittest', 'discover', '-s', 'experiment/tests',
          '-p', 'test_dataset_benchmark.py'])
stage('prepare-train')
display(json.loads((RUN_DIR / 'train_preparation.json').read_text()))

$ /usr/bin/python3 -m unittest discover -s experiment/tests -p test_dataset_benchmark.py
..........train UniProt canonical: attempted 1/1 missing accessions
train UniProt isoforms: reviewed 1/1; validated 1
.{
  "preprocessing": {
    "source_files": []
  },
  "validation": {
    "total": 2,
    "retained": 2,
    "coverage": 1.0,
    "reasons": {
      "validated": 2
    },
    "support": {
      "1": 1,
      "0": 1
    }
  },
  "coordinate_offset": 1,
  "training_coordinate_match_counts": {
    "0": 0,
    "1": 2
  },
  "cache_sha256": "bcca9d34710f24beae7c5d033870c171c2ae25443cce7577c8d5211e5152a0b4",
  "test_labels_used_for_selection": false,
  "isoform_resolution": {
    "status_counts": {
      "validated_candidate": 1
    },
    "unresolved_accessions": []
  },
  "coverage_policy": {
    "minimum": 0.95,
    "passed": true,
    "matched_cohort_only": false
  },
  "cohort_sha256": "b173c394386b1b591fc841f9e9ddfbb289bab82d2891ea1f531a9f85d51f949f"
}
train UniProt isoforms: review

{'cache_sha256': '5d00967b36edcf7ecb96032aef7d3ffd78f44c038dc400a91b6a284683064f9b',
 'cohort_sha256': 'daa5a79592707b5ec451915f7d5a6bc917ccd002f2f09d99f631d7bb2be5d625',
 'coordinate_offset': 1,
 'coverage_policy': {'matched_cohort_only': False,
  'minimum': 0.95,
  'passed': True},
 'preprocessing': {'exclusions': {},
  'preprocessed_sites': 5410,
  'raw_sites': 5410,
  'raw_support': {'0': 2738, '1': 2672},
  'source_files': [{'filename': 'UbiComb_training_data.txt',
    'sha256': '516206065e66dc7a9fd94e51d5657011924576ff75327197b48804364a1fc278'}]},
 'test_labels_used_for_selection': False,
 'training_coordinate_match_counts': {'0': 0, '1': 5402},
 'validation': {'coverage': 0.9985212569316081,
  'reasons': {'position_out_of_range': 6,
   'released_49mer_mismatch': 2,
   'validated': 5402},
  'retained': 5402,
  'support': {'0': 2736, '1': 2666},
  'total': 5410}}

## Five-fold development on two T4 GPUs

GPU 0 processes folds 0, 2 and 4; GPU 1 processes folds 1 and 3. Each worker writes to separate fold directories. Rerunning this cell reuses completed fold predictions and resumes incomplete candidates from the last completed epoch. When both workers finish, a final pass assembles all out-of-fold predictions and fits the residual fusion.

In [6]:
import threading, time

def run_parallel_development():
    assignments = [(0, [0, 2, 4]), (1, [1, 3])]
    processes, readers = [], []
    def stream_output(process, gpu):
        for line in process.stdout:
            print(f'[T4 {gpu}] {line}', end='', flush=True)
    try:
        for gpu, folds in assignments:
            env = os.environ.copy()
            env['CUDA_VISIBLE_DEVICES'] = str(gpu)
            env['OMP_NUM_THREADS'] = '2'
            env['TOKENIZERS_PARALLELISM'] = 'false'
            command = stage_command('develop', *[part for fold in folds
                                                  for part in ('--fold', str(fold))])
            print(f'Starting GPU {gpu} on folds {folds}', flush=True)
            process = subprocess.Popen([str(part) for part in command], env=env,
                                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                       text=True, bufsize=1)
            processes.append((gpu, process))
            reader = threading.Thread(target=stream_output, args=(process, gpu), daemon=True)
            reader.start()
            readers.append(reader)
        while True:
            failed = [(gpu, p.returncode) for gpu, p in processes
                      if p.poll() is not None and p.returncode != 0]
            if failed:
                raise RuntimeError(f'Fold worker failed: {failed}')
            if all(p.poll() == 0 for _, p in processes):
                break
            time.sleep(1)
    except BaseException:
        for _, process in processes:
            if process.poll() is None:
                process.terminate()
        for _, process in processes:
            process.wait()
        raise
    finally:
        for reader in readers:
            reader.join(timeout=5)

run_parallel_development()
stage('develop')
display(json.loads((RUN_DIR / 'development_summary.json').read_text()))

Starting GPU 0 on folds [0, 2, 4]
Starting GPU 1 on folds [1, 3]
[T4 0] fold=0 expert=local seed=42 budget=15
[T4 1] fold=1 expert=local seed=42 budget=15
[T4 0] epoch=001 started
[T4 1] epoch=001 started
[T4 0] epoch=001 loss=1.06319 val_mcc@0.5=0.36181 val_auprc=0.68626 mean_branch_positive_probabilities=[0.6893986463546753, 0.6730543375015259, 3.142629429930821e-05]
[T4 1] epoch=001 loss=1.05987 val_mcc@0.5=0.27136 val_auprc=0.57908 mean_branch_positive_probabilities=[0.4522014260292053, 0.8144555687904358, 0.0007791071548126638]
[T4 0] epoch=002 started
[T4 1] epoch=002 started
[T4 0] epoch=002 loss=0.71065 val_mcc@0.5=0.42002 val_auprc=0.72609 mean_branch_positive_probabilities=[0.6894229054450989, 0.7007937431335449, 5.8261261983716395e-06]
[T4 1] epoch=002 loss=0.71988 val_mcc@0.5=0.23107 val_auprc=0.61285 mean_branch_positive_probabilities=[0.5966820120811462, 0.7202857732772827, 8.858804108058393e-07]
[T4 1] epoch=003 started
[T4 0] epoch=003 started
[T4 1] epoch=003 loss=0.68

{'benchmark': 'ubicomb',
 'comparison_valid': True,
 'contract': {'cohort_sha256': 'daa5a79592707b5ec451915f7d5a6bc917ccd002f2f09d99f631d7bb2be5d625',
  'fold_sha256': '39083192b0c23795eb41aea28afe4fc876f4263099247d7aefda6edc8be7e892',
  'preparation_sha256': 'bada6caa0f594c1153905eb20a865607edab8727c4a557b5996163f32ce57484',
  'protocol_sha256': '97a61ee0b6aa2938dbe485ae1c2ca878fef3253c9eea326f870d2a2399f85782'},
 'crossfit_caveat': 'Fusion cross-fit over first-level OOF predictions; not fully nested end-to-end assessment',
 'fold_predictions_sha256': {'folds/fold_0/context/outer_predictions.npz': 'adeb9ce6fc4f44add71f4cd62228fdd9333de6de263f394cf80ff3d2e27dba16',
  'folds/fold_0/local/outer_predictions.npz': '511eeb354670965bec58af82d93c3c63bc5c553b68ad1246b07f60f5305cd816',
  'folds/fold_1/context/outer_predictions.npz': '29c52a6d7c2d10fe728f42c8e98075b279087d066bb508c86b7e3dd3750bbdfb',
  'folds/fold_1/local/outer_predictions.npz': 'ef15bd278ce803850f3f85d3500835900cb506b4b401147ac

## Full training-set refit and model lock

Each expert is refitted once on the eligible published training data for the median best epoch selected in development. The fusion weights and decision threshold are then frozen.

In [7]:
stage('refit')
display(json.loads((RUN_DIR / 'model_lock.json').read_text()))

$ /usr/bin/python3 -u experiment/scripts/run_dataset_benchmark.py refit --config experiment/configs/benchmarks/ubicomb_residual_v1.json --run-dir /kaggle/working/mmubipred_benchmark_runs/ubicomb/run --source-dir /kaggle/working/mmubipred_benchmark_sources/ubicomb --model-file /kaggle/working/mmubipred_benchmark_sources/ubicomb/UBICOMB_DeepUBI_AAindex_One_Hot_ProtT5_drop_out_Changed3257.h5
full refit expert=local epochs=13 sites=5402
refit_epoch=001/013 started
refit_epoch=001/013 loss=0.98848
refit_epoch=002/013 started
refit_epoch=002/013 loss=0.69166
refit_epoch=003/013 started
refit_epoch=003/013 loss=0.62761
refit_epoch=004/013 started
refit_epoch=004/013 loss=0.61279
refit_epoch=005/013 started
refit_epoch=005/013 loss=0.58860
refit_epoch=006/013 started
refit_epoch=006/013 loss=0.58102
refit_epoch=007/013 started
refit_epoch=007/013 loss=0.57149
refit_epoch=008/013 started
refit_epoch=008/013 loss=0.56017
refit_epoch=009/013 started
refit_epoch=009/013 loss=0.55613
refit_epoch=01

{'analysis_role': 'dataset-specific benchmark extension; architecture informed by prior PLMD and external experiments',
 'baseline_sha256': 'a1057c0e13d6dbd1348f83b26ed0fa347498dec914b7c671f274713fdc6cdefd',
 'benchmark': 'ubicomb',
 'checkpoints': {'context': '99fdbbbc361f7b434411dc4be56385a00dab87b8d1a4ac8b4af19ecd4864c58c',
  'local': 'bed5b5791dd28a2cbd691bc65efc252fc709d28509b2d6a3dd87906bec5ef06b'},
 'development_summary_sha256': 'a7c6feaffa63c7ea677282e841706f71faf843bffe8770d5e28c2dda54324a95',
 'protocol_sha256': '97a61ee0b6aa2938dbe485ae1c2ca878fef3253c9eea326f870d2a2399f85782',
 'refit_plan': {'context': {'epochs': 4, 'seed': 42},
  'local': {'epochs': 13, 'seed': 42}},
 'stacker': {'context_weight': 0.3127730094719311,
  'intercept': -0.05540489618257199,
  'l2_strength': 0.01,
  'local_weight': 0.7567563081282915},
 'test_inference_performed': False,
 'threshold': 0.5,
 'training_cohort_sha256': 'daa5a79592707b5ec451915f7d5a6bc917ccd002f2f09d99f631d7bb2be5d625',
 'training

## Published test preparation

Run only after the model lock exists. Both the released checkpoint and the newly trained hybrid will be evaluated on the same retained, context-valid test sites.

In [8]:
assert (RUN_DIR / 'model_lock.json').exists(), 'Complete the refit first.'
stage('prepare-test', '--allow-test')
display(json.loads((RUN_DIR / 'test_preparation.json').read_text()))

$ /usr/bin/python3 -u experiment/scripts/run_dataset_benchmark.py prepare-test --config experiment/configs/benchmarks/ubicomb_residual_v1.json --run-dir /kaggle/working/mmubipred_benchmark_runs/ubicomb/run --source-dir /kaggle/working/mmubipred_benchmark_sources/ubicomb --model-file /kaggle/working/mmubipred_benchmark_sources/ubicomb/UBICOMB_DeepUBI_AAindex_One_Hot_ProtT5_drop_out_Changed3257.h5 --allow-test
test UniProt canonical: attempted 40/412 missing accessions
test UniProt canonical: attempted 80/412 missing accessions
test UniProt canonical: attempted 120/412 missing accessions
test UniProt canonical: attempted 160/412 missing accessions
test UniProt canonical: attempted 200/412 missing accessions
test UniProt canonical: attempted 240/412 missing accessions
test UniProt canonical: attempted 280/412 missing accessions
test UniProt canonical: attempted 320/412 missing accessions
test UniProt canonical: attempted 360/412 missing accessions
test UniProt canonical: attempted 400/412

{'cache_sha256': 'ea212dc5e407e69e02a4e13e524b5ff161af7442b0c962157170200a986231f2',
 'cohort_sha256': 'c30ad4c3a4a211f1b37d7b9672fc620f436cd429e0d935804ace6300473e2ab2',
 'coordinate_offset': 1,
 'coverage_policy': {'matched_cohort_only': False,
  'minimum': 0.95,
  'passed': True},
 'overlap': {'policy': 'published split preserved; overlap reported, not silently filtered',
  'test_sites_with_training_49mer': 1,
  'test_sites_with_training_protein': 453,
  'test_sites_with_training_site': 0},
 'preprocessing': {'exclusions': {},
  'preprocessed_sites': 1490,
  'raw_sites': 1490,
  'raw_support': {'0': 747, '1': 743},
  'source_files': [{'filename': 'Positive_UbiCom_testing.fasta',
    'sha256': '7489012cf71b7173bc6e26264aed6cce531101dd494c95e4acbdd208a38719af'},
   {'filename': 'Negative_UbiCom_testing.fasta',
    'sha256': '8d7dc133fcf70e70c87c845977270088b2ed99ec72e4c0093ad56f4718f7fd0a'}]},
 'test_labels_used_for_selection': False,
 'training_coordinate_match_counts': None,
 'valid

## Legacy Keras comparator

The published HDF5 checkpoint uses legacy Keras layers. Install the matching `tf-keras` package without replacing Kaggle's existing TensorFlow dependencies.

In [9]:
import importlib.metadata as metadata

try:
    tf_version = metadata.version('tensorflow')
except metadata.PackageNotFoundError as error:
    raise RuntimeError('TensorFlow is absent from this Kaggle image; install a compatible TensorFlow CPU package before evaluation.') from error
legacy_requirement = 'tf-keras==' + '.'.join(tf_version.split('.')[:2]) + '.*'
run_live([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', legacy_requirement])

$ /usr/bin/python3 -m pip install -q --no-deps tf-keras==2.20.*


## Matched test comparison

The evaluation uses the locked hybrid, the authors' dataset-specific checkpoint, fixed threshold 0.5 and protein-cluster paired bootstrap. Keep the output together with the run directory.

In [10]:
stage('evaluate', '--allow-test')
result = json.loads((RUN_DIR / 'test_results.json').read_text())
display(result['metrics'])
display(result['paired_hybrid_vs_paper'])

$ /usr/bin/python3 -u experiment/scripts/run_dataset_benchmark.py evaluate --config experiment/configs/benchmarks/ubicomb_residual_v1.json --run-dir /kaggle/working/mmubipred_benchmark_runs/ubicomb/run --source-dir /kaggle/working/mmubipred_benchmark_sources/ubicomb --model-file /kaggle/working/mmubipred_benchmark_sources/ubicomb/UBICOMB_DeepUBI_AAindex_One_Hot_ProtT5_drop_out_Changed3257.h5 --allow-test

Loading weights: 100%|██████████| 198/198 [00:00<00:00, 4038.73it/s]
[transformers] EsmModel LOAD REPORT from: facebook/esm2_t12_35M_UR50D
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-10-12 22:20

{'context': {'accuracy': 0.6550335570469799,
  'auprc': 0.6883830637081239,
  'auroc': 0.7161602894304901,
  'confusion_matrix': [[450, 297], [217, 526]],
  'f1': 0.6717752234993615,
  'mcc': 0.3120643870083124,
  'precision': 0.6391251518833536,
  'sensitivity': 0.7079407806191117,
  'specificity': 0.6024096385542169,
  'support': {'negative': 747, 'positive': 743, 'total': 1490},
  'threshold': 0.5},
 'hybrid': {'accuracy': 0.7644295302013423,
  'auprc': 0.7883996036652925,
  'auroc': 0.83405132418413,
  'confusion_matrix': [[469, 278], [73, 670]],
  'f1': 0.7924305144884684,
  'mcc': 0.5504200685768197,
  'precision': 0.7067510548523207,
  'sensitivity': 0.901749663526245,
  'specificity': 0.6278447121820616,
  'support': {'negative': 747, 'positive': 743, 'total': 1490},
  'threshold': 0.5},
 'local': {'accuracy': 0.7550335570469798,
  'auprc': 0.7975466514169057,
  'auroc': 0.8362944825511107,
  'confusion_matrix': [[448, 299], [66, 677]],
  'f1': 0.7876672484002327,
  'mcc': 0.53

{'confidence_level': 0.95,
 'method': 'paired nonparametric cluster bootstrap of candidate-minus-reference metric differences; whole canonical-accession protein groups resampled with replacement',
 'metrics': {'accuracy': {'candidate': 0.7644295302013423,
   'confidence_interval': [-0.11021872048758162, -0.04864487843210267],
   'difference': -0.08120805369127515,
   'paired_bootstrap_two_sided_p_value': 0.00019998000199980003,
   'reference': 0.8456375838926175},
  'auprc': {'candidate': 0.7883996036652925,
   'confidence_interval': [-0.13584348575321373, -0.049642177039259425],
   'difference': -0.09223266133299268,
   'paired_bootstrap_two_sided_p_value': 0.00019998000199980003,
   'reference': 0.8806322649982852},
  'auroc': {'candidate': 0.83405132418413,
   'confidence_interval': [-0.09396367002078727, -0.05068111053450959],
   'difference': -0.07199006884424186,
   'paired_bootstrap_two_sided_p_value': 0.00019998000199980003,
   'reference': 0.9060413930283718},
  'f1': {'candid

## Resumption and output transfer

With **Persistence → Files only**, rerun setup, paths and the interrupted stage after a session restart. Completed fold predictions are skipped; an interrupted epoch is repeated. For a different Kaggle account, save or export the entire `/kaggle/working/mmubipred_benchmark_runs/ubicomb/run` directory, attach it to the new notebook as Input, and set `RESUME_RUN_DIR` in the paths cell. Use the same repository commit printed during the original run; changing the training implementation invalidates the recorded protocol. The public checkpoint can be downloaded again.

Kaggle's `/kaggle/working` files are not guaranteed to survive every failure. Save a notebook version with outputs or download a run snapshot before the session ends. The two-T4 parallelism applies to development folds; refit and final inference are single-process stages.